# Analyze MarketACD Run

## Imports

In [ ]:
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from util import load_config
from artifact_manager import ArtifactManager
from train import setup_environment

## Setup

In [ ]:
config_file = "config/default_marketacd.yaml"
#config_file = "config/default_faustmann.yaml"
seed = 0


In [ ]:
config = load_config(config_file)
# Create an artifact manager to handle the paths cleanly.
artifact_manager = ArtifactManager(config, seed)

def get_artifact_path(identifier, file_name, artifact_manager=artifact_manager):
    get_loss_path = artifact_manager.register_artifact(
        artifact_identifier=identifier,
        artifact_description="MarketACD_foobar",
        file_name_generator=lambda x: file_name,
        check_conflicts=False
    )
    return lambda x=None, ensure_dir=False: get_loss_path(x, ensure_dir=ensure_dir)

# Load losses.
# -----------------------------------------------------------------------------
losses_path = get_artifact_path("training_losses", "training_losses.csv")
losses_df = pd.read_csv(losses_path())
print(f"Loading losses from: {losses_path()}")

# Load distribution parameters.
# -----------------------------------------------------------------------------
distribution_path = get_artifact_path("distribution_parameters", "distribution_parameters.csv")
distribution_df = pd.read_csv(distribution_path())
print(f"Loading distribution parameters from: {distribution_path()}")

# Load visit frequencies.
# -----------------------------------------------------------------------------
visit_freq_path = get_artifact_path("visit_frequencies", "visit_frequencies.csv")
visit_freq_df = pd.read_csv(visit_freq_path())
print(f"Loading visit frequencies from: {visit_freq_path()}")

# Load return estimates.
# -----------------------------------------------------------------------------
return_estimate_path = get_artifact_path("return_estimates", "return_estimates.csv")
return_estimates_df = pd.read_csv(return_estimate_path())
print(f"Loading return estimates from: {return_estimate_path()}")

## Plot

### Plot Losses

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot critic loss.
axes[0].plot(losses_df.index, losses_df["critic_loss"], linewidth=1.5, color="blue")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Loss")
axes[0].set_title("Critic Loss", fontweight="bold")
axes[0].grid(True, alpha=0.3)

# Plot actor loss.
axes[1].plot(losses_df.index, losses_df["actor_loss"], linewidth=1.5, color="red")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Loss")
axes[1].set_title("Actor Loss", fontweight="bold")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
#plt.save("loss.png")
plt.show()

### Plot Distribution Parameters

In [ ]:
# Detect distribution type from column names.
# Check if columns start with "loc" (logistic normal) or "A" (beta).
is_logistic_normal = any(col.startswith("loc") for col in distribution_df.columns)

if is_logistic_normal:
    # Logistic normal distribution.
    param1_columns = [col for col in distribution_df.columns if col.startswith("loc")]
    param2_columns = [col for col in distribution_df.columns if col.startswith("scale")]
    param1_label = "$\\mu$ (loc)"
    param2_label = "$\\sigma$ (scale)"
    dist_name = "Logistic-Normal"
else:
    # Beta distribution.
    param1_columns = [col for col in distribution_df.columns if col.startswith("A")]
    param2_columns = [col for col in distribution_df.columns if col.startswith("B")]
    param1_label = "$\\alpha$"
    param2_label = "$\\beta$"
    dist_name = "Beta"

# Create figure with three rows.
n_param1_cols = len(param1_columns)
n_param2_cols = len(param2_columns)
max_cols = max(n_param1_cols, n_param2_cols)
fig, axes = plt.subplots(3, max_cols, figsize=(4 * max_cols, 12))

# Plot first parameter in the first row.
for i, col in enumerate(param1_columns):
    ax = axes[0, i]
    ax.plot(distribution_df.index, distribution_df[col], linewidth=1.5)
    ax.set_xlabel("Episode")
    ax.set_ylabel(param1_label)
    ax.set_title(f"{col}", fontweight="bold")
    if not is_logistic_normal:
        ax.set_ylim(0, 5)
    ax.grid(True, alpha=0.3)

# Plot second parameter in the second row.
for i, col in enumerate(param2_columns):
    ax = axes[1, i]
    ax.plot(distribution_df.index, distribution_df[col], linewidth=1.5)
    ax.set_xlabel("Episode")
    ax.set_ylabel(param2_label)
    ax.set_title(f"{col}", fontweight="bold")
    if not is_logistic_normal:
        ax.set_ylim(0, 5)
    ax.grid(True, alpha=0.3)

# Plot mean actions in the third row.
for i, (col_1, col_2) in enumerate(zip(param1_columns, param2_columns)):
    ax = axes[2, i]
    if is_logistic_normal:
        # For logistic normal: mean ≈ sigmoid(loc) for small scale.
        loc = distribution_df[col_1]
        mean = 1.0 / (1.0 + np.exp(-loc))
    else:
        # For beta: mean = alpha / (alpha + beta).
        A = distribution_df[col_1]
        B = distribution_df[col_2]
        mean = A / (A + B)
    
    ax.plot(distribution_df.index, mean, linewidth=1.5)
    ax.set_xlabel("Episode")
    ax.set_ylabel("Mean Action")
    ax.set_title(f"Age Class {i+1} Mean Action", fontweight="bold")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)

plt.suptitle(f"{dist_name} Distribution Parameters", fontsize=16, y=1.001)
plt.tight_layout()
#plt.save("distribution_output.png")
plt.show()

### Plot Visit Frequencies

In [ ]:
# Convert to numpy array for plotting.
data = visit_freq_df.values
n_age_classes = data.shape[0]
n_bins = data.shape[1]

# Create figure with subplots for each age class.
n_cols = 5
n_rows = int(np.ceil(n_age_classes / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
axes = axes.flatten()

# Create histogram for each age class.
bin_edges = np.arange(n_bins + 1)  # Bin edges for the histogram.

for i in range(n_age_classes):
    ax = axes[i]
    # Plot histogram - the values in the row represent frequencies for each bin.
    ax.bar(bin_edges[:-1], data[i, :], width=0.8, align="edge", alpha=0.7, edgecolor="black")
    ax.set_xlabel("Bin", fontsize=10)
    ax.set_ylabel("Frequency", fontsize=10)
    ax.set_title(f"Age Class {i+1}", fontweight="bold")
    ax.grid(True, alpha=0.3, axis="y")
    ax.set_xticks(bin_edges)

# Remove extra subplots.
for i in range(n_age_classes, len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
#plt.save("visit_frequencies.png")
plt.show()

### Plot Return Estimates

In [ ]:
# Plot return estimates over episodes
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(return_estimates_df.index, return_estimates_df["return_estimate"], linewidth=1.5, color="green")
ax.set_xlabel("Episode", fontsize=12)
ax.set_ylabel("Estimated State Value", fontsize=12)
ax.set_title("Return Estimates for Fixed State Over Training", fontsize=14, fontweight="bold")
ax.grid(True, alpha=0.3)

plt.tight_layout()
#plt.save("return_estimates.png")
plt.show()


# Play Around with the Environment

In [ ]:
# Create one environment.
tmp = copy.deepcopy(config)
tmp["environment"]["N_env"] = 1
env = setup_environment(tmp)
discount_factor = env.get_attr("discount_factor")[0]

In [ ]:
# Initialize the environment with a given state.
init_state = np.array([1/6]*6 + [0.0]*9, dtype=np.float32)
#init_state = env.call("get_random_state")[0]
state, _ = env.reset(options=[{"reset_state": init_state}])
print(f"Initialized the environment with state {init_state}")

In [ ]:
# Set up long time return computation.
total_return = 0.0

In [ ]:
# Perform an action and observe the result.
action = np.array([0.0]*5 + [0.2] + [0.6] + [1.0]*8, dtype=np.float32)
next_state, reward, terminated, _, _ = env.step([action])
total_return = reward[0] + discount_factor * total_return
print("Action", action)
print("Next state", list([float(x) for x in next_state[0]]))
print("Reward", reward[0])
print("Return", total_return)